# Notebook para Análise de Dados com PokeAPI

- **Objetivo**.

Nesta etapa, o(a) candidato(a) deverá consumir dados diretamente da API pública da PokeAPI e
realizar análises utilizando Apache Spark.O objetivo é avaliar habilidades de:

    - Ingestão de dados a partir de API REST
    - Tratamento de paginação e dados aninhados (JSON)
    - Modelagem relacional
    - Manipulação e agregação de dados com Spark
    - Clareza na organização e documentação do código

In [12]:
import pathlib
import asyncio
import aiohttp
from aiohttp import ClientSession
from typing import List, Dict, Any
import json
import logging
from tenacity import retry, wait_random_exponential, stop_after_attempt, retry_if_exception_type

from pyspark.sql import SparkSession
from pyspark.sql import functions as Fsql
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
)

logging.basicConfig(level=logging.INFO)

BASE_URL = "https://pokeapi.co/api/v2/pokemon/"
CONCURRENT_REQUESTS_LIMIT = 10

## Etapa 1 — Data Extraction

Consumir o endpoint /pokemon para obter a listagem completa de todos os pokémons.

1. Para cada url retornada, realizar uma nova requisição e extrair exclusivamente os seguintes
campos e transformá los em novas tabelas:
types, stats, abilities

In [2]:
class RetryForHTTPError(Exception):
    """Custom exception to trigger retry for HTTP errors."""
    def __init__(self, status_code: int, url: str):
        self.status_code = status_code
        self.url = url
        super().__init__(f"HTTP error {status_code} for URL: {url}")

class PokemonFetchData:
    def __init__(self, base_url: str, concurrent_requests_limit: int = 10):
        self.base_url = base_url
        self.semaphore = asyncio.Semaphore(concurrent_requests_limit)

    @retry(retry=retry_if_exception_type((aiohttp.ClientError,
                                          RetryForHTTPError,
                                          asyncio.TimeoutError)),
           wait=wait_random_exponential(multiplier=1, max=10), 
           stop=stop_after_attempt(3),
           reraise=True
           )
    async def _fetch_url_json(self,
                            session: aiohttp.ClientSession, 
                            url: str,
                            semaphore: asyncio.Semaphore
                            ) -> Dict[str, Any]:
        """ Internal Method to fetch JSON data from a given URL using aiohttp with retry logic.
        Args:
            session (aiohttp.ClientSession): The aiohttp session to use for the request.
            url (str): The URL to fetch data from.
            semaphore (asyncio.Semaphore): Semaphore to limit concurrent requests.
        Returns:
            Dict[str, Any]: The JSON data fetched from the URL, or an empty dictionary in case of an error.
        """
        async with semaphore:
            
            async with session.get(url) as response:
                #error
                if response.status == 429 or 500<=response.status<600:
                    raise RetryForHTTPError(response.status, url)
                
                response.raise_for_status()
                return await response.json()
            


    async def _fetch_all_pokemons_base_urls(self) -> List[Dict[str, Any]]:
        """Internal Method to fetch all pokemon URLs from the PokeAPI.
        Returns:
            List[Dict[str, Any]]: A list of dictionaries containing the base URLs of all pokemons.
        """
        timeout = aiohttp.ClientTimeout(total=30)
        all_pokemon_data = []
        try:
            async with aiohttp.ClientSession(timeout=timeout) as session:
                data = await self._fetch_url_json(session, self.base_url, self.semaphore)
                all_pokemon_data.extend(data.get("results", []))

                while data.get("next"):
                    next_url = data["next"]
                    data = await self._fetch_url_json(session, next_url, self.semaphore)
                    all_pokemon_data.extend(data.get("results", []))
        except (aiohttp.ClientError, asyncio.TimeoutError) as e:
            logging.error(f"Error fetching all pokemons: {e}")
        return all_pokemon_data


    async def _fetch_pokemon_raw_data(self,
                                     pokemon_urls_list:List[Dict[str,Any]]
                                     ) -> Dict[str, Any]:
        """ Internal Method to fetch each pokemon types, stats, and abilities from the PokeAPI using the list of pokemon URLs.
        
        Args:
            pokemon_urls_list (List[Dict[str, Any]]): List of dictionaries containing pokemon names and their corresponding URLs.
        Returns:
            Dict[str, Any]: A list of dictionaries containing pokemon names and their corresponding types, stats, abilities, height, weight, and base_experience data.
        """
        async with aiohttp.ClientSession() as session:
            try:
                data = [self._fetch_url_json(session, pokemon["url"], self.semaphore) for pokemon in pokemon_urls_list]
                pokemon_fetch_data = await asyncio.gather(*data)
            except Exception as e:
                logging.error(f"Error fetching pokemon raw data: {e}")

            pokemons_raw_data = [ 
                {"name": pokemon["name"],
                "pokemon_id": payload["id"],
                "height": payload["height"],
                "weight": payload["weight"],
                "base_experience": payload["base_experience"],
                "types": payload["types"], 
                "stats": payload["stats"], 
                "abilities": payload["abilities"]} 
                for pokemon, payload in zip(pokemon_urls_list, pokemon_fetch_data) if payload
                ]
            return pokemons_raw_data


    def create_tables(self, pokemons_full_raw_data: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
        """Function to create tables for each pokemon with their corresponding stats.
        
        Args:
            pokemons_full_raw_data (List[Dict[str, Any]]): A list of dictionaries containing pokemon names and their corresponding status data.
        Returns:
            Dict[str, List[Dict[str, Any]]]: A dictionary containing tables for each pokemon with their corresponding stats.
        """
        pokemon_table = []
        pokemon_type_table = []
        pokemon_stats_table = []
        pokemon_ability_table = []

        

    async def fetch_pokemons(self, pokemons_raw_data_address: str):
        """Function to fetch all pokemons from the PokeAPI and save their raw data to a JSON file.

        Args:
            pokemons_raw_data_address (str): The file path to save the raw pokemon data.
        """
        pokemons_base_data = await self._fetch_all_pokemons_base_urls()

        pokemons_raw_data = await self._fetch_pokemon_raw_data(pokemons_base_data)

        file_path = pathlib.Path(pokemons_raw_data_address).parent
        file_path.mkdir(parents=True, exist_ok=True)

        with open(pokemons_raw_data_address, "w", encoding="utf-8") as f:
            json.dump(pokemons_raw_data, f, indent=2, ensure_ascii=False)

        return pokemons_raw_data



In [3]:
pokemon_object = PokemonFetchData(base_url=BASE_URL, concurrent_requests_limit=CONCURRENT_REQUESTS_LIMIT)
pokemons_raw_data_address = "data/raw/pokemons_raw_data.json"
pokemons_raw_data = await pokemon_object.fetch_pokemons(pokemons_raw_data_address)

In [21]:
def create_tables(pokemons_raw_data: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    """Function to create tables for each pokemon with their corresponding stats.
    
    Args:
        pokemons_raw_data (List[Dict[str, Any]]): A list of dictionaries containing pokemon names and their corresponding status data.
    Returns:
        Dict[str, List[Dict[str, Any]]]: A dictionary containing tables for each pokemon with their corresponding stats.
    """
    pokemon_table = []
    pokemon_type_table = []
    pokemon_stats_table = []
    pokemon_ability_table = []

    for pokemon in pokemons_raw_data:
        pokemon_table.append({
            "name": pokemon["name"],
            "pokemon_id": pokemon["pokemon_id"],
            "height": pokemon["height"],
            "weight": pokemon["weight"],
            "base_experience": pokemon["base_experience"]
        })

        for type_info in pokemon["types"]:
            pokemon_type_table.append({
                "pokemon_id": pokemon["pokemon_id"],
                "type_name": type_info["type"]["name"]
            })

        for stat_info in pokemon["stats"]:
            pokemon_stats_table.append({
                "pokemon_id": pokemon["pokemon_id"],
                "stat_name": stat_info["stat"]["name"],
                "base_stat": stat_info["base_stat"]
            })

        for ability_info in pokemon["abilities"]:
            pokemon_ability_table.append({
                "pokemon_id": pokemon["pokemon_id"],
                "ability_name": ability_info["ability"]["name"],
                "is_hidden": ability_info["is_hidden"]
            })
    return {
        "pokemon": pokemon_table,
        "pokemon_types": pokemon_type_table,
        "pokemon_stats": pokemon_stats_table,
        "pokemon_abilities": pokemon_ability_table
    }


In [22]:

spark = (
    SparkSession.builder
    .appName("pokemon-case")
    .master("local[*]")
    .getOrCreate()
)

pokemon_schema = StructType([
    StructField("pokemon_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("height", IntegerType(), True),
    StructField("weight", IntegerType(), True),
    StructField("base_experience", IntegerType(), True),
])

pokemon_types_schema = StructType([
    StructField("pokemon_id", IntegerType(), False),
    StructField("type_name", StringType(), False),
])

pokemon_stats_schema = StructType([
    StructField("pokemon_id", IntegerType(), False),
    StructField("stat_name", StringType(), False),
    StructField("base_stat", IntegerType(), True),
])

pokemon_abilities_schema = StructType([
    StructField("pokemon_id", IntegerType(), False),
    StructField("ability_name", StringType(), False),
    StructField("is_hidden", BooleanType(), True),
])

In [23]:
with open(pokemons_raw_data_address, "r", encoding="utf-8") as f:
    pokemons_raw_data = json.load(f)

created_tables = create_tables(pokemons_raw_data)

pokemon_df = spark.createDataFrame(created_tables["pokemon"], schema=pokemon_schema)
pokemon_types_df = spark.createDataFrame(created_tables["pokemon_types"], schema=pokemon_types_schema)
pokemon_stats_df = spark.createDataFrame(created_tables["pokemon_stats"], schema=pokemon_stats_schema)
pokemon_abilities_df = spark.createDataFrame(created_tables["pokemon_abilities"], schema=pokemon_abilities_schema)

pokemon_df.write.mode("overwrite").parquet("data/trusted/pokemon.parquet")
pokemon_types_df.write.mode("overwrite").parquet("data/trusted/pokemon_types.parquet")
pokemon_stats_df.write.mode("overwrite").parquet("data/trusted/pokemon_stats.parquet")
pokemon_abilities_df.write.mode("overwrite").parquet("data/trusted/pokemon_abilities.parquet")

## Etapa 3 — Data Analysis using Spark

Após a construção das tabelas, o(a) candidato(a) deverá responder às seguintes perguntas utilizando
Apache Spark:
Caso você não tenha um ambiente Spark, tente ver o Databricks Community Edition.

#### 3.1. Pokémons com múltiplos tipos e força acima da média

Definição de força: soma de todos os base_stat de cada pokémon.
Pergunta:
Quantos pokémons possuem mais de um type_name e têm a força maior que a média geral de todos os
pokémons?

In [46]:
# 1.1 Calcular a força média de todos os pokémons
pokemons_mean_strength = (pokemon_stats_df.groupBy("pokemon_id").sum("base_stat").agg({"sum(base_stat)": "avg"})).collect()[0][0]


# Ids de pokemon com mais de um tipo.
pokemons_single_type = (pokemon_types_df
                        .groupBy("pokemon_id")
                        .count()
                        .filter(Fsql.col("count") > 1)
                        .select("pokemon_id"))

# 1.2 Filtrar quais pokemons tem mais de um tipo (Count type_name) e calcular quantos deles possuem força acima da média
pokemons_above_mean_strength_count = (pokemon_stats_df
                                      .join(pokemons_single_type, on="pokemon_id")
                                      .groupBy("pokemon_id").agg(
                                          Fsql.sum("base_stat").alias("total_strength")
                                      )).filter(Fsql.col("total_strength") > pokemons_mean_strength).count()

pokemons_above_mean_strength_count



510

#### 3.2. Abilities exclusivas de pokémons com múltiplos tipos

Pergunta:
Quais habilidades não aparecem em nenhum pokémon de tipo único?
Queremos identificar aquelas habilidades que só existem em pokémons que têm mais de um tipo.
Exemplo :

Habilidade levitate → só aparece em pokémons com 2 tipos ✅ \
Habilidade overgrow → aparece em pokémons com 1 tipo ❌

In [49]:
# listagem unica do total de habilidades de todos os pokemons
pokemons_total_habilities = (pokemon_abilities_df.select("ability_name").distinct())


# Ids de pokemon com apenas um tipo.
pokemons_single_type = (pokemon_types_df
                        .groupBy("pokemon_id")
                        .count()
                        .filter(Fsql.col("count") == 1)
                        .select("pokemon_id"))

## Habilidades de pokemons que possuem apenas um tipo
pokemons_single_type_habilities = (pokemon_abilities_df
                                   .join(pokemons_single_type, on="pokemon_id")
                                   .select("ability_name")
                                   .distinct())

## Diferença entre as duas listagens
pokemons_diff_habilities = pokemons_total_habilities.subtract(pokemons_single_type_habilities)


In [37]:
print(f" Número de habilidades diferentes que só existem em Pokemons com mais de uma habilidade: {pokemons_diff_habilities}")

 Número de habilidades diferentes que só existem em Pokemons com mais de uma habilidade: 93


#### 3.3. Pokémons mais versáteis

Pergunta: Quais são os 5 pokémons que apresentam maior versatilidade de acordo com este critério?
A métrica de cálculo estabelecida para o índice de versatilidade é a seguinte:
versatility_score = (número de tipos * 2) + (número de abilities) + (soma
dos stats / 100)
Exemplo:
● Pokémon X → Pontuação = 14.2
● Pokémon Y → Pontuação = 1

In [83]:
# Ids de pokemon com apenas um tipo.
types_count_df = (pokemon_types_df
                        .groupBy("pokemon_id")
                        .count()).withColumn("types_count", Fsql.col("count")*2)

abilities_count_df = pokemon_abilities_df.groupBy("pokemon_id").agg(Fsql.count("pokemon_id").alias("habilities_count"))

stats_sum_df = (pokemon_stats_df.groupBy("pokemon_id")
                .agg(Fsql.sum("base_stat").alias("total_strength"))).withColumn("percentage_strength", Fsql.col("total_strength") / 100)

result_df = (types_count_df
             .join(abilities_count_df, on="pokemon_id")
             .join(stats_sum_df, on="pokemon_id")
             ).withColumn("versatility_score", 
                          (Fsql.col("types_count") + Fsql.col("habilities_count") + Fsql.col("percentage_strength"))
                          ).select("pokemon_id", "versatility_score").orderBy(Fsql.col("versatility_score").desc()).limit(5)

In [ ]:

result_df.show(5)

+----------+-----------------+
|pokemon_id|versatility_score|
+----------+-----------------+
|     10190|            16.25|
|       784|             13.0|
|       887|             13.0|
|      1018|             13.0|
|     10146|             13.0|
+----------+-----------------+



In [87]:
print(f"Resultado: Os cinco pokémons mais versáteis são: \n{result_df.collect()}")
# result_df.show(5)

Resultado: Os cinco pokémons mais versáteis são: 
[Row(pokemon_id=10190, versatility_score=16.25), Row(pokemon_id=784, versatility_score=13.0), Row(pokemon_id=887, versatility_score=13.0), Row(pokemon_id=1018, versatility_score=13.0), Row(pokemon_id=10146, versatility_score=13.0)]
